In [8]:
import pandas as pd

fighters = pd.read_csv('raw_fighter_details.csv')
fights = pd.read_csv('raw_total_fight_data.csv')

# Clean Fight Data File

In [9]:
import pandas as pd
import numpy as np
# Input/output paths
input_path = "raw_total_fight_data.csv"
output_path = "clean_total_fight_data.csv"

# Read semicolon-separated raw file
raw_fights = pd.read_csv(
    input_path,
    sep=";",
    keep_default_na=False
)

# Clean column names
raw_fights.columns = raw_fights.columns.str.strip()

# Clean string cells only
for col in raw_fights.select_dtypes(include="object").columns:
    raw_fights[col] = raw_fights[col].str.strip()

# Save as standard comma-separated CSV
raw_fights.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", raw_fights.shape)
print(raw_fights.head())

Saved: clean_total_fight_data.csv
Shape: (6012, 41)
       R_fighter          B_fighter  R_KD  B_KD R_SIG_STR. B_SIG_STR.  \
0   Adrian Yanez      Gustavo Lopez     2     0  41 of 103   23 of 51   
1   Trevin Giles      Roman Dolidze     0     0   27 of 57   32 of 67   
2    Tai Tuivasa    Harry Hunsucker     1     0   14 of 18     2 of 6   
3  Cheyanne Buys  Montserrat Conejo     0     0   31 of 65   15 of 41   
4  Marion Reneau      Macy Chiasson     0     0   30 of 63  51 of 138   

  R_SIG_STR_pct B_SIG_STR_pct R_TOTAL_STR. B_TOTAL_STR.  ... B_GROUND  \
0           39%           45%    41 of 103     23 of 51  ...   0 of 0   
1           47%           47%     43 of 73    75 of 110  ...   1 of 2   
2           77%           33%     14 of 18       2 of 6  ...   0 of 0   
3           47%           36%     49 of 87   136 of 168  ...   0 of 0   
4           47%           36%     59 of 93    92 of 184  ...   2 of 4   

                 win_by last_round last_round_time         Format  \
0

C:\Users\Abtin\AppData\Local\Temp\ipykernel_21608\2338410796.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in raw_fights.select_dtypes(include="object").columns:


In [10]:
# Replace "---" in takedown percentage columns with "0%"
raw_fights["R_TD_pct"] = raw_fights["R_TD_pct"].replace("---", "0%")
raw_fights["B_TD_pct"] = raw_fights["B_TD_pct"].replace("---", "0%")

# Convert percentage columns from strings like "45%" to decimals like 0.45
pct_cols = [
    "R_SIG_STR_pct",
    "B_SIG_STR_pct",
    "R_TD_pct",
    "B_TD_pct"
]

for col in pct_cols:
    raw_fights[col] = (
        raw_fights[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .replace("---", "0")
        .astype(float) / 100
    )

# Drop unnecessary columns
cols_to_drop = ["Referee", "date", "location", "Fight_type"]

raw_fights = raw_fights.drop(columns=cols_to_drop)

# Save updated cleaned file
raw_fights.to_csv("clean_total_fight_data.csv", index=False)

print("Saved: clean_total_fight_data.csv")
print("Shape:", raw_fights.shape)
print(raw_fights[pct_cols].head())

Saved: clean_total_fight_data.csv
Shape: (6012, 37)
   R_SIG_STR_pct  B_SIG_STR_pct  R_TD_pct  B_TD_pct
0           0.39           0.45       0.0      0.00
1           0.47           0.47       0.5      0.33
2           0.77           0.33       0.0      0.00
3           0.47           0.36       0.0      0.80
4           0.47           0.36       0.5      1.00


In [16]:
# Split fight-level rows into fighter-level rows, including opponent stats for defense

shared_cols = ["Winner", "win_by", "last_round", "last_round_time", "Format"]

stat_cols = [
    "KD",
    "SIG_STR.",
    "SIG_STR_pct",
    "TOTAL_STR.",
    "TD",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "CTRL",
    "HEAD",
    "BODY",
    "LEG",
    "DISTANCE",
    "CLINCH",
    "GROUND"
]

opp_stat_cols = [
    "SIG_STR.",
    "TOTAL_STR.",
    "TD",
    "HEAD",
    "BODY",
    "LEG",
    "DISTANCE",
    "CLINCH",
    "GROUND"
]

def opp_name(col):
    return col.replace(".", "") + "_opp"


# Red fighter rows
r_rows = raw_fights[
    ["R_fighter", "B_fighter"] 
    + shared_cols 
    + [f"R_{col}" for col in stat_cols]
    + [f"B_{col}" for col in opp_stat_cols]
].copy()

r_rows = r_rows.rename(columns={
    "R_fighter": "fighter_name",
    "B_fighter": "opponent",
    **{f"R_{col}": col for col in stat_cols},
    **{f"B_{col}": opp_name(col) for col in opp_stat_cols}
})


# Blue fighter rows
b_rows = raw_fights[
    ["B_fighter", "R_fighter"] 
    + shared_cols 
    + [f"B_{col}" for col in stat_cols]
    + [f"R_{col}" for col in opp_stat_cols]
].copy()

b_rows = b_rows.rename(columns={
    "B_fighter": "fighter_name",
    "R_fighter": "opponent",
    **{f"B_{col}": col for col in stat_cols},
    **{f"R_{col}": opp_name(col) for col in opp_stat_cols}
})


# Combine red and blue rows
fighter_fights = pd.concat([r_rows, b_rows], ignore_index=True)


# Create result column
fighter_fights["result"] = np.where(
    fighter_fights["Winner"].eq(""),
    "Draw",
    np.where(
        fighter_fights["fighter_name"].eq(fighter_fights["Winner"]),
        "Win",
        "Loss"
    )
)


# Reorder columns
ordered_cols = [
    "fighter_name",
    "result",
    "win_by",
    "last_round",
    "last_round_time",
    "Format",

    "KD",
    "SIG_STR.",
    "SIG_STR_pct",
    "TOTAL_STR.",
    "TD",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "CTRL",
    "HEAD",
    "BODY",
    "LEG",
    "DISTANCE",
    "CLINCH",
    "GROUND",

    "SIG_STR_opp",
    "TOTAL_STR_opp",
    "TD_opp",
    "HEAD_opp",
    "BODY_opp",
    "LEG_opp",
    "DISTANCE_opp",
    "CLINCH_opp",
    "GROUND_opp"
]

fighter_fights = fighter_fights[ordered_cols]


# Sort by fighter name
fighter_fights = fighter_fights.sort_values(
    by=["fighter_name"],
    ascending=True
).reset_index(drop=True)


# Save
fighter_fights.to_csv("clean_fighter_fight_data.csv", index=False)

print("Saved: clean_fighter_fight_data.csv")
print("Shape:", fighter_fights.shape)
print(fighter_fights.head())

Saved: clean_fighter_fight_data.csv
Shape: (12024, 30)
     fighter_name result                win_by  last_round last_round_time  \
0      Aalon Cruz   Loss                KO/TKO           1            1:40   
1      Aalon Cruz   Loss                KO/TKO           1            1:25   
2     Aaron Brink   Loss            Submission           1            0:55   
3  Aaron Phillips   Loss            Submission           2            2:29   
4  Aaron Phillips   Loss  Decision - Unanimous           3            5:00   

          Format  KD  SIG_STR.  SIG_STR_pct  TOTAL_STR.  ...  GROUND  \
0  3 Rnd (5-5-5)   0    0 of 3         0.00      0 of 3  ...  0 of 0   
1  3 Rnd (5-5-5)   0   2 of 12         0.16     2 of 12  ...  0 of 0   
2  3 Rnd (5-5-5)   0    0 of 5         0.00      0 of 5  ...  0 of 0   
3  3 Rnd (5-5-5)   0   6 of 15         0.40     7 of 16  ...  0 of 0   
4  3 Rnd (5-5-5)   0  18 of 40         0.45  109 of 137  ...  6 of 8   

   SIG_STR_opp  TOTAL_STR_opp  TD_opp  HEAD

In [42]:
for x in list(fighter_fights.columns):
    print(x)
    print(fighter_fights[x].unique())

fighter_name
<StringArray>
[          'Aalon Cruz',          'Aaron Brink',       'Aaron Phillips',
          'Aaron Riley',           'Aaron Rosa',        'Aaron Simpson',
      'Aaron Wilkinson', 'Abdul Razak Alhassan',   'Abdul-Kerim Edilov',
        'Abel Trujillo',
 ...
            'Zak Ottow',         'Zane Frazier',          'Zarah Fairn',
      'Zarrukh Adashev',        'Zelim Imadaev',   'Zhalgas Zhumagulov',
         'Zhang Lipeng',        'Zhang Tiequan',          'Zhang Weili',
     'Zubaira Tukhugov']
Length: 2139, dtype: str
result
<StringArray>
['Loss', 'Win', 'Draw']
Length: 3, dtype: str
win_by
<StringArray>
[                 'KO/TKO',              'Submission',
    'Decision - Unanimous', 'TKO - Doctor's Stoppage',
        'Decision - Split',     'Decision - Majority',
      'Could Not Continue',                      'DQ',
              'Overturned',                   'Other']
Length: 10, dtype: str
last_round
[1 2 3 5 4]
last_round_time
<StringArray>
[ '1:40',  '1:25

In [17]:
# 1. Group win_by into broader categories

win_by_map = {
    "KO/TKO": "KO_TKO",
    "TKO - Doctor's Stoppage": "KO_TKO",
    "Could Not Continue": "KO_TKO",

    "Submission": "Submission",

    "Decision - Unanimous": "Decision",
    "Decision - Split": "Decision",
    "Decision - Majority": "Decision"
}

fighter_fights["win_by"] = fighter_fights["win_by"].map(win_by_map).fillna("Other")


# 2. Convert last_round_time from mm:ss to seconds

def time_to_seconds(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()

    if x == "":
        return 0

    parts = x.split(":")

    if len(parts) == 2:
        minutes = int(parts[0])
        seconds = int(parts[1])
        return minutes * 60 + seconds

    return 0


fighter_fights["last_round_time"] = fighter_fights["last_round_time"].apply(time_to_seconds)


# 3. Convert Format into total_scheduled_time

def format_to_total_scheduled_time(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()

    if x == "No Time Limit":
        return 30*60

    if "(" not in x or ")" not in x:
        return 0

    inside_parentheses = x.split("(")[1].split(")")[0]

    round_lengths = inside_parentheses.split("-")

    total_minutes = sum(int(length) for length in round_lengths)

    return total_minutes*60


fighter_fights["total_scheduled_time"] = fighter_fights["Format"].apply(format_to_total_scheduled_time)

# Drop original Format column after extracting total scheduled time
fighter_fights = fighter_fights.drop(columns=["Format"])


# 4. Convert CTRL from mm:ss to seconds

fighter_fights["CTRL"] = fighter_fights["CTRL"].apply(time_to_seconds)


# 5. Split columns like "42 of 68" into landed and attempted

count_cols = [
    "SIG_STR.",
    "TOTAL_STR.",
    "TD",
    "HEAD",
    "BODY",
    "LEG",
    "DISTANCE",
    "CLINCH",
    "GROUND"
]

opp_count_cols = [
    "SIG_STR_opp",
    "TOTAL_STR_opp",
    "TD_opp",
    "HEAD_opp",
    "BODY_opp",
    "LEG_opp",
    "DISTANCE_opp",
    "CLINCH_opp",
    "GROUND_opp"
]

def split_landed_attempted(x):
    if pd.isna(x):
        return pd.Series([0, 0])

    x = str(x).strip()

    if "of" not in x:
        return pd.Series([0, 0])

    landed, attempted = x.split("of")

    return pd.Series([
        int(landed.strip()),
        int(attempted.strip())
    ])

# Split fighter's own count columns
for col in count_cols:
    new_col_base = col.replace(".", "")

    fighter_fights[[f"{new_col_base}_landed", f"{new_col_base}_attempted"]] = (
        fighter_fights[col].apply(split_landed_attempted)
    )

# Split opponent count columns
for col in opp_count_cols:
    new_col_base = col.replace("_opp", "")

    fighter_fights[[f"{new_col_base}_opp_landed", f"{new_col_base}_opp_attempted"]] = (
        fighter_fights[col].apply(split_landed_attempted)
    )

# Drop original text columns
fighter_fights = fighter_fights.drop(columns=count_cols + opp_count_cols)

# Save result
fighter_fights.to_csv("clean_fighter_fight_data.csv", index=False)

print("Saved: clean_fighter_fight_data.csv")
print("Shape:", fighter_fights.shape)

print(fighter_fights.head())

Saved: clean_fighter_fight_data.csv
Shape: (12024, 48)
     fighter_name result      win_by  last_round  last_round_time  KD  \
0      Aalon Cruz   Loss      KO_TKO           1              100   0   
1      Aalon Cruz   Loss      KO_TKO           1               85   0   
2     Aaron Brink   Loss  Submission           1               55   0   
3  Aaron Phillips   Loss  Submission           2              149   0   
4  Aaron Phillips   Loss    Decision           3              300   0   

   SIG_STR_pct  TD_pct  SUB_ATT  REV  ...  BODY_opp_landed  \
0         0.00     0.0        0    0  ...                1   
1         0.16     0.0        0    0  ...                0   
2         0.00     0.0        0    0  ...                0   
3         0.40     0.0        0    0  ...                5   
4         0.45     0.0        1    1  ...                8   

   BODY_opp_attempted  LEG_opp_landed  LEG_opp_attempted  DISTANCE_opp_landed  \
0                   1               1               

In [18]:
# Create total fight time in seconds
# Assumption: completed rounds before the final round are 5 minutes each

fighter_fights["total_fight_time"] = (
    (fighter_fights["last_round"] - 1) * 5 * 60
    + fighter_fights["last_round_time"]
)

# Save
fighter_fights.to_csv("clean_fighter_fight_data.csv", index=False)

print("Saved: clean_fighter_fight_data.csv")
print(
    fighter_fights[
        ["fighter_name", "last_round", "last_round_time", "total_fight_time"]
    ].head()
)

Saved: clean_fighter_fight_data.csv
     fighter_name  last_round  last_round_time  total_fight_time
0      Aalon Cruz           1              100               100
1      Aalon Cruz           1               85                85
2     Aaron Brink           1               55                55
3  Aaron Phillips           2              149               449
4  Aaron Phillips           3              300               900


In [19]:
# Consolidate fighter_fights into one row per fighter

def safe_divide(num, denom):
    return np.where(denom > 0, num / denom, 0)

def safe_divide_nan(num, denom):
    return np.where(denom > 0, num / denom, np.nan)


# Normalize result and win_by values
fighter_fights["result"] = fighter_fights["result"].astype(str).str.strip()
fighter_fights["win_by"] = fighter_fights["win_by"].astype(str).str.strip()

win_by_clean = {
    "Decision": "DEC",
    "KO_TKO": "KO_TKO",
    "Submission": "SUB"
}

fighter_fights["win_by_clean"] = fighter_fights["win_by"].map(win_by_clean).fillna("Other")


# Create method indicators
fighter_fights["win_DEC"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "DEC")).astype(int)
fighter_fights["win_KO_TKO"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "KO_TKO")).astype(int)
fighter_fights["win_SUB"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "SUB")).astype(int)

fighter_fights["loss_DEC"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "DEC")).astype(int)
fighter_fights["loss_KO_TKO"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "KO_TKO")).astype(int)
fighter_fights["loss_SUB"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "SUB")).astype(int)

fighter_fights["is_win"] = (fighter_fights["result"] == "Win").astype(int)
fighter_fights["is_loss"] = (fighter_fights["result"] == "Loss").astype(int)


# Make sure numeric columns are numeric
numeric_cols = [
    "total_fight_time", "total_scheduled_time", "CTRL",
    "KD", "SUB_ATT", "REV",

    "SIG_STR_landed", "SIG_STR_attempted",
    "TOTAL_STR_landed", "TOTAL_STR_attempted",
    "TD_landed", "TD_attempted",

    "HEAD_landed", "HEAD_attempted",
    "BODY_landed", "BODY_attempted",
    "LEG_landed", "LEG_attempted",
    "DISTANCE_landed", "DISTANCE_attempted",
    "CLINCH_landed", "CLINCH_attempted",
    "GROUND_landed", "GROUND_attempted",

    "SIG_STR_opp_landed", "SIG_STR_opp_attempted",
    "TOTAL_STR_opp_landed", "TOTAL_STR_opp_attempted",
    "TD_opp_landed", "TD_opp_attempted",

    "HEAD_opp_landed", "HEAD_opp_attempted",
    "BODY_opp_landed", "BODY_opp_attempted",
    "LEG_opp_landed", "LEG_opp_attempted",
    "DISTANCE_opp_landed", "DISTANCE_opp_attempted",
    "CLINCH_opp_landed", "CLINCH_opp_attempted",
    "GROUND_opp_landed", "GROUND_opp_attempted"
]

for col in numeric_cols:
    fighter_fights[col] = pd.to_numeric(fighter_fights[col], errors="coerce").fillna(0)


# Aggregate totals by fighter
agg_cols = {
    "total_fight_time": "sum",
    "total_scheduled_time": "sum",
    "CTRL": "sum",

    "is_win": "sum",
    "is_loss": "sum",
    "win_DEC": "sum",
    "win_KO_TKO": "sum",
    "win_SUB": "sum",
    "loss_DEC": "sum",
    "loss_KO_TKO": "sum",
    "loss_SUB": "sum",

    "KD": "sum",
    "SUB_ATT": "sum",
    "REV": "sum",

    "SIG_STR_landed": "sum",
    "SIG_STR_attempted": "sum",
    "TOTAL_STR_landed": "sum",
    "TOTAL_STR_attempted": "sum",
    "TD_landed": "sum",
    "TD_attempted": "sum",

    "HEAD_landed": "sum",
    "HEAD_attempted": "sum",
    "BODY_landed": "sum",
    "BODY_attempted": "sum",
    "LEG_landed": "sum",
    "LEG_attempted": "sum",
    "DISTANCE_landed": "sum",
    "DISTANCE_attempted": "sum",
    "CLINCH_landed": "sum",
    "CLINCH_attempted": "sum",
    "GROUND_landed": "sum",
    "GROUND_attempted": "sum",

    "SIG_STR_opp_landed": "sum",
    "SIG_STR_opp_attempted": "sum",
    "TOTAL_STR_opp_landed": "sum",
    "TOTAL_STR_opp_attempted": "sum",
    "TD_opp_landed": "sum",
    "TD_opp_attempted": "sum",

    "HEAD_opp_landed": "sum",
    "HEAD_opp_attempted": "sum",
    "BODY_opp_landed": "sum",
    "BODY_opp_attempted": "sum",
    "LEG_opp_landed": "sum",
    "LEG_opp_attempted": "sum",
    "DISTANCE_opp_landed": "sum",
    "DISTANCE_opp_attempted": "sum",
    "CLINCH_opp_landed": "sum",
    "CLINCH_opp_attempted": "sum",
    "GROUND_opp_landed": "sum",
    "GROUND_opp_attempted": "sum"
}

fighter_career = fighter_fights.groupby("fighter_name").agg(agg_cols).reset_index()
fighter_career["total_fights"] = fighter_fights.groupby("fighter_name").size().values

fighter_career = fighter_career.rename(columns={"fighter_name": "fighter"})

total_minutes = fighter_career["total_fight_time"] / 60


# Fight quickness
fighter_career["fight_quickness"] = safe_divide(
    fighter_career["total_fight_time"],
    fighter_career["total_scheduled_time"]
)


# Win/loss method percentages
fighter_career["win_DEC_pct"] = safe_divide(fighter_career["win_DEC"], fighter_career["is_win"])
fighter_career["win_KO_TKO_pct"] = safe_divide(fighter_career["win_KO_TKO"], fighter_career["is_win"])
fighter_career["win_SUB_pct"] = safe_divide(fighter_career["win_SUB"], fighter_career["is_win"])

fighter_career["loss_DEC_pct"] = safe_divide(fighter_career["loss_DEC"], fighter_career["is_loss"])
fighter_career["loss_KO_TKO_pct"] = safe_divide(fighter_career["loss_KO_TKO"], fighter_career["is_loss"])
fighter_career["loss_SUB_pct"] = safe_divide(fighter_career["loss_SUB"], fighter_career["is_loss"])


# Control percentage
fighter_career["CTRL_pct"] = safe_divide(
    fighter_career["CTRL"],
    fighter_career["total_fight_time"]
)


# Offensive accuracy columns
fighter_career["SIG_STR_acc"] = safe_divide(
    fighter_career["SIG_STR_landed"],
    fighter_career["SIG_STR_attempted"]
)

fighter_career["TD_acc"] = safe_divide(
    fighter_career["TD_landed"],
    fighter_career["TD_attempted"]
)

fighter_career["TOTAL_STR_acc"] = safe_divide(
    fighter_career["TOTAL_STR_landed"],
    fighter_career["TOTAL_STR_attempted"]
)


# Offensive per-minute rates
fighter_career["KD_per_min"] = safe_divide(fighter_career["KD"], total_minutes)
fighter_career["SUB_ATT_per_min"] = safe_divide(fighter_career["SUB_ATT"], total_minutes)
fighter_career["REV_per_min"] = safe_divide(fighter_career["REV"], total_minutes)

fighter_career["SIG_STR_land_per_min"] = safe_divide(fighter_career["SIG_STR_landed"], total_minutes)
fighter_career["SIG_STR_att_per_min"] = safe_divide(fighter_career["SIG_STR_attempted"], total_minutes)

fighter_career["TOTAL_STR_land_per_min"] = safe_divide(fighter_career["TOTAL_STR_landed"], total_minutes)
fighter_career["TOTAL_STR_att_per_min"] = safe_divide(fighter_career["TOTAL_STR_attempted"], total_minutes)

fighter_career["TD_land_per_min"] = safe_divide(fighter_career["TD_landed"], total_minutes)
fighter_career["TD_att_per_min"] = safe_divide(fighter_career["TD_attempted"], total_minutes)

fighter_career["HEAD_land_per_min"] = safe_divide(fighter_career["HEAD_landed"], total_minutes)
fighter_career["HEAD_att_per_min"] = safe_divide(fighter_career["HEAD_attempted"], total_minutes)

fighter_career["BODY_land_per_min"] = safe_divide(fighter_career["BODY_landed"], total_minutes)
fighter_career["BODY_att_per_min"] = safe_divide(fighter_career["BODY_attempted"], total_minutes)

fighter_career["LEG_land_per_min"] = safe_divide(fighter_career["LEG_landed"], total_minutes)
fighter_career["LEG_att_per_min"] = safe_divide(fighter_career["LEG_attempted"], total_minutes)

fighter_career["DISTANCE_land_per_min"] = safe_divide(fighter_career["DISTANCE_landed"], total_minutes)
fighter_career["DISTANCE_att_per_min"] = safe_divide(fighter_career["DISTANCE_attempted"], total_minutes)

fighter_career["CLINCH_land_per_min"] = safe_divide(fighter_career["CLINCH_landed"], total_minutes)
fighter_career["CLINCH_att_per_min"] = safe_divide(fighter_career["CLINCH_attempted"], total_minutes)

fighter_career["GROUND_land_per_min"] = safe_divide(fighter_career["GROUND_landed"], total_minutes)
fighter_career["GROUND_att_per_min"] = safe_divide(fighter_career["GROUND_attempted"], total_minutes)


# Defensive absorbed and opponent attempt rates
fighter_career["SIG_STR_absorbed_per_min"] = safe_divide(fighter_career["SIG_STR_opp_landed"], total_minutes)
fighter_career["SIG_STR_opp_att_per_min"] = safe_divide(fighter_career["SIG_STR_opp_attempted"], total_minutes)
fighter_career["SIG_STR_def"] = 1 - safe_divide(
    fighter_career["SIG_STR_opp_landed"],
    fighter_career["SIG_STR_opp_attempted"]
)

fighter_career["TOTAL_STR_absorbed_per_min"] = safe_divide(fighter_career["TOTAL_STR_opp_landed"], total_minutes)
fighter_career["TOTAL_STR_opp_att_per_min"] = safe_divide(fighter_career["TOTAL_STR_opp_attempted"], total_minutes)
fighter_career["TOTAL_STR_def"] = 1 - safe_divide(
    fighter_career["TOTAL_STR_opp_landed"],
    fighter_career["TOTAL_STR_opp_attempted"]
)

fighter_career["TD_absorbed_per_min"] = safe_divide(fighter_career["TD_opp_landed"], total_minutes)
fighter_career["TD_opp_att_per_min"] = safe_divide(fighter_career["TD_opp_attempted"], total_minutes)
fighter_career["TD_def"] = 1 - safe_divide_nan(
    fighter_career["TD_opp_landed"],
    fighter_career["TD_opp_attempted"]
)

fighter_career["HEAD_absorbed_per_min"] = safe_divide(fighter_career["HEAD_opp_landed"], total_minutes)
fighter_career["HEAD_opp_att_per_min"] = safe_divide(fighter_career["HEAD_opp_attempted"], total_minutes)

fighter_career["BODY_absorbed_per_min"] = safe_divide(fighter_career["BODY_opp_landed"], total_minutes)
fighter_career["BODY_opp_att_per_min"] = safe_divide(fighter_career["BODY_opp_attempted"], total_minutes)

fighter_career["LEG_absorbed_per_min"] = safe_divide(fighter_career["LEG_opp_landed"], total_minutes)
fighter_career["LEG_opp_att_per_min"] = safe_divide(fighter_career["LEG_opp_attempted"], total_minutes)

fighter_career["DISTANCE_absorbed_per_min"] = safe_divide(fighter_career["DISTANCE_opp_landed"], total_minutes)
fighter_career["DISTANCE_opp_att_per_min"] = safe_divide(fighter_career["DISTANCE_opp_attempted"], total_minutes)

fighter_career["CLINCH_absorbed_per_min"] = safe_divide(fighter_career["CLINCH_opp_landed"], total_minutes)
fighter_career["CLINCH_opp_att_per_min"] = safe_divide(fighter_career["CLINCH_opp_attempted"], total_minutes)

fighter_career["GROUND_absorbed_per_min"] = safe_divide(fighter_career["GROUND_opp_landed"], total_minutes)
fighter_career["GROUND_opp_att_per_min"] = safe_divide(fighter_career["GROUND_opp_attempted"], total_minutes)


# Final column order
final_cols = [
    "fighter",
    "total_fights",
    "total_fight_time",
    "fight_quickness",

    "win_DEC_pct",
    "win_KO_TKO_pct",
    "win_SUB_pct",
    "loss_DEC_pct",
    "loss_KO_TKO_pct",
    "loss_SUB_pct",

    "CTRL_pct",
    "SIG_STR_acc",
    "TD_acc",
    "TOTAL_STR_acc",

    "KD_per_min",
    "SUB_ATT_per_min",
    "REV_per_min",

    "SIG_STR_land_per_min",
    "SIG_STR_att_per_min",

    "SIG_STR_absorbed_per_min",
    "SIG_STR_opp_att_per_min",
    "SIG_STR_def",

    "TOTAL_STR_land_per_min",
    "TOTAL_STR_att_per_min",

    "TOTAL_STR_absorbed_per_min",
    "TOTAL_STR_opp_att_per_min",
    "TOTAL_STR_def",

    "TD_land_per_min",
    "TD_att_per_min",

    "TD_absorbed_per_min",
    "TD_opp_att_per_min",
    "TD_def",

    "HEAD_land_per_min",
    "HEAD_att_per_min",

    "HEAD_absorbed_per_min",
    "HEAD_opp_att_per_min",

    "BODY_land_per_min",
    "BODY_att_per_min",

    "BODY_absorbed_per_min",
    "BODY_opp_att_per_min",

    "LEG_land_per_min",
    "LEG_att_per_min",

    "LEG_absorbed_per_min",
    "LEG_opp_att_per_min",

    "DISTANCE_land_per_min",
    "DISTANCE_att_per_min",

    "DISTANCE_absorbed_per_min",
    "DISTANCE_opp_att_per_min",

    "CLINCH_land_per_min",
    "CLINCH_att_per_min",

    "CLINCH_absorbed_per_min",
    "CLINCH_opp_att_per_min",

    "GROUND_land_per_min",
    "GROUND_att_per_min",

    "GROUND_absorbed_per_min",
    "GROUND_opp_att_per_min"
]

fighter_career_stats = fighter_career[final_cols]

fighter_career_stats.to_csv("fighter_career_stats.csv", index=False)

print("Saved: fighter_career_stats.csv")
print("Shape:", fighter_career_stats.shape)
print(fighter_career_stats.head())

Saved: fighter_career_stats.csv
Shape: (2139, 56)
          fighter  total_fights  total_fight_time  fight_quickness  \
0      Aalon Cruz             2               185         0.102778   
1     Aaron Brink             1                55         0.061111   
2  Aaron Phillips             3              2249         0.832963   
3     Aaron Riley             9              5722         0.706420   
4      Aaron Rosa             3              1714         0.634815   

   win_DEC_pct  win_KO_TKO_pct  win_SUB_pct  loss_DEC_pct  loss_KO_TKO_pct  \
0          0.0             0.0          0.0      0.000000         1.000000   
1          0.0             0.0          0.0      0.000000         0.000000   
2          0.0             0.0          0.0      0.666667         0.000000   
3          1.0             0.0          0.0      0.333333         0.666667   
4          1.0             0.0          0.0      0.000000         1.000000   

   loss_SUB_pct  ...  DISTANCE_absorbed_per_min  DISTANCE_op

C:\Users\Abtin\AppData\Local\Temp\ipykernel_21608\1912392032.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fighter_career["CLINCH_absorbed_per_min"] = safe_divide(fighter_career["CLINCH_opp_landed"], total_minutes)
C:\Users\Abtin\AppData\Local\Temp\ipykernel_21608\1912392032.py:245: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fighter_career["CLINCH_opp_att_per_min"] = safe_divide(fighter_career["CLINCH_opp_attempted"], total_minutes)
C:\Users\Abtin\AppData\Local\Temp\ipykernel_21608\1912392032.py:247: PerformanceWarnin